In [8]:
import os
import re
import csv
import json
import pickle
import random
import torch
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from copy import deepcopy
from typing import Union, Tuple, Dict
from collections import defaultdict, Counter
from transformers import AutoTokenizer

In [9]:
def pickle_load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def pickle_dump(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def json_load(path):
    with open(path, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def json_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def jsonl_load(path):
    data = []
    with open(path, mode='r', encoding='utf-8') as f:
        # lines = f.readline()
        # lines = json.loads(lines)
        # for line in lines:
        #     data.append(line)
        
        for line in f:
            data.append(json.loads(line))
    return data

In [10]:
data = jsonl_load('../data/QA_Datasets/2wiki/dev.jsonl')
len(data)

12576

In [13]:
qa_list = []
cache = {}
idx = 0
contents2docid = {}

for sample in data:
    qid = sample['id']
    question = sample['question']
    answer = [str(ans) for ans in sample['golden_answers']]
    question_type = sample['metadata']['type']
            
    titles = sample['metadata']['context']['title']
    docs = sample['metadata']['context']['content']
    assert len(titles) == len(docs)

    new_docs = []
    for title, doc in zip(titles, docs):
        contents = ' '.join(doc)
        if len(contents) == 0:
            continue
        docid = contents2docid[contents] if contents in contents2docid else idx
        if docid == idx:
            idx += 1
        new_docs.append({'id':docid, 'contents':contents})
        
    if len(new_docs) == 0:
        continue

    cache[question] = new_docs
    
    qa_list.append({
        'id': qid, 
        'Question': question, 
        'answer': answer, 
        'type': question_type, 
    })

len(qa_list), len(cache)

(12576, 12576)

In [14]:
type2qids = defaultdict(list)
for qa in qa_list:
    qid = qa['id']
    question_type = qa['type']
    type2qids[question_type].append(qid)

for k, v in type2qids.items():
    print(f'{k}: {len(v)}')

compositional: 5236
comparison: 3040
inference: 1549
bridge_comparison: 2751


In [15]:
path = '/data/Search-o1/data/QA_Datasets/2wiki'
json_dump(os.path.join(path, '2wiki_full.json'), qa_list)

path = '/data/Search-o1/cache/2wiki'
json_dump(os.path.join(path, 'search_cache_full.json'), cache)

In [16]:
random.seed(42)
random.shuffle(qa_list)
sampled_qa_list = qa_list[:500]
sampled_cache = {}
for qa in sampled_qa_list:
    question = qa['Question']
    sampled_cache[question] = deepcopy(cache[question])
len(sampled_qa_list), len(sampled_cache)

(500, 500)

In [18]:
type2qids = defaultdict(list)
for qa in sampled_qa_list:
    qid = qa['id']
    question_type = qa['type']
    type2qids[question_type].append(qid)

for k, v in type2qids.items():
    print(f'{k}: {len(v)}')

bridge_comparison: 101
comparison: 123
inference: 59
compositional: 217


In [17]:
path = '/data/Search-o1/data/QA_Datasets/2wiki'
json_dump(os.path.join(path, '2wiki_500.json'), sampled_qa_list)

path = '/data/Search-o1/cache/2wiki'
json_dump(os.path.join(path, 'search_cache_500.json'), sampled_cache)